In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from keras.src.layers import Dense, GlobalMaxPooling2D, Dropout
from keras.src.models import Sequential
from keras.src.callbacks import EarlyStopping
from keras.src.applications.resnet import ResNet50
from keras.src.applications.mobilenet_v3 import MobileNetV3Small
import sys
sys.path.append("../Handlers")
import preprocessing
from functools import partial
import url_preprocessing
from PIL import Image
import os
import joblib

[nltk_data] Downloading package punkt to E:/nltk...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
from concurrent.futures import ThreadPoolExecutor

class MultimodalClassifier:
    def __init__(self, text_model=None, image_model=None, url_model=None):
        self.text_model = text_model
        self.image_model = image_model
        self.url_model = url_model

    def train(self,
              threading=False, 
              text_data=None, 
              text_labels=None, 
              image_data=None, 
              image_labels=None, 
              url_data=None, 
              url_labels=None):
        def train_text():        
            if text_data is not None and text_labels is not None:
                print("Begin training text")
                self.build_text_model()

                X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
                    text_data, text_labels, test_size=0.15, random_state=42
                )

                self.text_model.fit(X_text_train, y_text_train)
                text_pred = self.text_model.predict(X_text_test)
                text_accuracy = accuracy_score(y_text_test, text_pred)
                print(f"Text model accuracy: {text_accuracy:.4f}")

                del X_text_train, X_text_test, y_text_train, y_text_test

        def train_image():
            if image_data is not None and image_labels is not None:
                print("Begin training image")
                self.build_image_model()

                X_image_train, X_image_test, y_image_train, y_image_test = train_test_split(
                    image_data, image_labels, test_size=0.15, random_state=42
                )

                history = self.image_model.fit(
                    X_image_train, y_image_train,
                    validation_data=(X_image_test, y_image_test),
                    epochs=10, batch_size=32, verbose=1,
                    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
                )

                img_pred = self.image_model.predict(X_image_test)
                img_pred_classes = (img_pred > 0.5).astype(int)
                img_accuracy = accuracy_score(y_image_test, img_pred_classes)
                print(f"Image model accuracy: {img_accuracy:.4f}")

                del X_image_train, X_image_test, y_image_train, y_image_test

        def train_url():
            if url_data is not None and url_labels is not None:
                print("Begin training URL")
                self.build_url_model()

                X_url_train, X_url_test, y_url_train, y_url_test = train_test_split(
                    url_data, url_labels, test_size=0.2, random_state=42
                )

                self.url_model.fit(X_url_train, y_url_train)
                url_pred = self.url_model.predict(X_url_test)
                url_accuracy = accuracy_score(y_url_test, url_pred)
                print(f"URL model accuracy: {url_accuracy:.4f}")

        if threading:
            executor = ThreadPoolExecutor(max_workers=2)
            executor.submit(train_text)
            executor.submit(train_image)
            executor.submit(train_url)

            executor.shutdown()
        else:
            train_text()
            train_image()
            train_url()

    def predict(self, text_data=None, image_data=None, url_data=None):
        predictions = []

        if text_data is not None and self.text_model is not None:
            text_pred = self.text_model.predict_proba(text_data)
            predictions.append(text_pred)

        if image_data is not None and self.image_model is not None:
            img_pred = self.image_model.predict(image_data)
            # img_pred_classes = (img_pred > 0.5).astype(int)
            predictions.append(img_pred)

        if url_data is not None and self.url_model is not None:
            url_pred = self.url_model.predict_proba(url_data)
            predictions.append(url_pred)

        stacked_preds = np.stack(predictions, axis=0)

        return (np.mean(stacked_preds, axis=1) > 0.5).astype(int)
    
    def preprocess_text_data(self, text_data):
        preprocession = partial(
            preprocessing.preprocess_text,
            remove_numbers=True
        )
        if isinstance(text_data, pd.DataFrame):
            text_data["preprocessed"] = text_data.apply(preprocession)

            text_data = text_data.apply(preprocessing.lemmatizing)

            self.tfidf = TfidfVectorizer(min_df=3)
            text_data = self.tfidf.fit_transform(text_data)

        return text_data
    
    def preprocess_url_data(self, url_data:pd.DataFrame):
        url_data, _ = url_preprocessing.cleanup_url_dataset(url_data)

        url_data = url_preprocessing.extract_url_features(url_data)

        return url_data
    
    def preprocess_image_data(self, image_0_path, image_1_path, save=True, X_save_path="./image_data/data_X.npy",
            y_save_path="./image_data/data_y.npy",
            img_size=(224, 224)):
        
        print("Loading and preprocessing data...")
        
        data = []
        
        # Load spam images (label 1)
        for filename in os.listdir(image_1_path):
            path = os.path.join(image_1_path, filename)
            try:
                img = Image.open(path).convert('RGB').resize(img_size)
                data.append((np.array(img), 1))
            except Exception as e:
                print(f"Error loading {filename}: {e}")
        
        # Load natural images (label 0)
        for filename in os.listdir(image_0_path):
            path = os.path.join(image_0_path, filename)
            try:
                img = Image.open(path).convert('RGB').resize(img_size)
                data.append((np.array(img), 0))
            except Exception as e:
                print(f"Error loading {filename}: {e}")
        
        X, y = zip(*data)

        if save:
            print("Saving np data")
            np.save(X_save_path, X)
            np.save(y_save_path, y)
        
        print(f"Loaded {len(X)} images")
        
        return X, y

    def build_text_model(self):
        self.text_model = VotingClassifier([
            ('rf', RandomForestClassifier(random_state=42, n_jobs=4)),
            ('ext', ExtraTreesClassifier(n_jobs=4, random_state=42)),
            ('svc', BaggingClassifier(n_jobs=4, random_state=42))
        ], voting="soft", n_jobs=2)

    def build_url_model(self):
        self.url_model = VotingClassifier([
            ('rf', RandomForestClassifier(random_state=42, n_jobs=4)),
            ('ext', ExtraTreesClassifier(n_jobs=4, random_state=42)),
            ('svc', BaggingClassifier(n_jobs=4, random_state=42))
        ], voting="soft", n_jobs=2)

    def build_image_model(self):
        base_model = MobileNetV3Small(
            include_top=False,
            input_shape=(224, 224, 3)
        )
        
        self.image_model = Sequential([
            base_model,
            GlobalMaxPooling2D(),
            Dense(128, activation="relu"),
            Dropout(0.4),
            Dense(64, activation='relu'),
            Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])

        self.image_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import random
from collections import Counter

class MultimodalDatasetMixer:
    def __init__(self, text_csv_path, url_csv_path, image_folder_path, target_size=1739):
        """
        Initialize the dataset mixer
        
        Args:
            text_csv_path: Path to text CSV file
            url_csv_path: Path to URL CSV file  
            image_folder_path: Path to folder containing images
            target_size: Number of samples for mixed datasets (default: 1739)
        """
        self.text_csv_path = text_csv_path
        self.url_csv_path = url_csv_path
        self.image_folder_path = image_folder_path
        self.target_size = target_size
        
        # Load datasets
        self.text_df = None
        self.url_df = None
        self.image_files = []
        
    def load_datasets(self, text_col='text', text_label_col='label', 
                     url_col='url', url_label_col='label'):
        """Load and prepare individual datasets"""
        
        print("Loading text dataset...")
        self.text_df = pd.read_csv(self.text_csv_path)
        try:
            self.text_df["text"] = self.text_df.apply(lambda x: f'{x["Subject"]} {x["Body"]}', axis=1)
        except:
            print("Error merging columns")
        print(f"Text dataset: {len(self.text_df)} samples")
        print(f"Text label distribution: {Counter(self.text_df[text_label_col])}")
        
        print("\nLoading URL dataset...")
        self.url_df = pd.read_csv(self.url_csv_path)
        print(f"URL dataset: {len(self.url_df)} samples")
        print(f"URL label distribution: {Counter(self.url_df[url_label_col])}")
        
        print("\nLoading image files...")
        self.image_files = []
        for root, dirs, files in os.walk(self.image_folder_path):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                    # Assume folder structure: spam/ and ham/ or similar
                    label = os.path.basename(root).lower()
                    if label in ['spam', 'ham']:
                        self.image_files.append({
                            'path': os.path.join(root, file),
                            'label': label
                        })
        
        print(f"Image dataset: {len(self.image_files)} samples")
        image_labels = [img['label'] for img in self.image_files]
        print(f"Image label distribution: {Counter(image_labels)}")
        
        # Store column names for later use
        self.text_col = text_col
        self.text_label_col = text_label_col
        self.url_col = url_col
        self.url_label_col = url_label_col
    
    def stratified_sample(self, df, label_col, n_samples):
        """Create stratified sample maintaining label distribution"""
        label_counts = df[label_col].value_counts()
        label_ratios = label_counts / len(df)
        
        sampled_dfs = []
        for label, ratio in label_ratios.items():
            label_df = df[df[label_col] == label]
            n_label_samples = int(n_samples * ratio)
            
            # Handle case where we need more samples than available
            if n_label_samples > len(label_df):
                # Sample with replacement
                sampled_df = label_df.sample(n=n_label_samples, replace=True, random_state=42)
            else:
                sampled_df = label_df.sample(n=n_label_samples, random_state=42)
            
            sampled_dfs.append(sampled_df)
        
        return pd.concat(sampled_dfs, ignore_index=True).sample(frac=1, random_state=42)
    
    def stratified_sample_images(self, n_samples):
        """Create stratified sample of images"""
        image_df = pd.DataFrame(self.image_files)
        return self.stratified_sample_list(self.image_files, 'label', n_samples)
    
    def stratified_sample_list(self, items_list, label_key, n_samples):
        """Stratified sampling for list of dictionaries"""
        # Convert to DataFrame for easier manipulation
        df = pd.DataFrame(items_list)
        sampled_df = self.stratified_sample(df, label_key, n_samples)
        return sampled_df.to_dict('records')
    
    def generate_mixed_datasets(self):
        """Generate all possible combinations of mixed datasets"""
        if not all([self.text_df is not None, self.url_df is not None, self.image_files]):
            raise ValueError("Please load datasets first using load_datasets()")
        
        datasets = {}
        
        print(f"\nGenerating mixed datasets with {self.target_size} samples each...")
        
        # 1. Single modality datasets
        print("Creating single modality datasets...")
        
        # Text only
        text_sample = self.stratified_sample(self.text_df, self.text_label_col, self.target_size)
        datasets['text_only'] = {
            'text': text_sample[self.text_col].tolist(),
            'url': [None] * len(text_sample),
            'image': [None] * len(text_sample),
            'labels': text_sample[self.text_label_col].tolist()
        }
        
        # URL only
        url_sample = self.stratified_sample(self.url_df, self.url_label_col, self.target_size)
        datasets['url_only'] = {
            'text': [None] * len(url_sample),
            'url': url_sample[self.url_col].tolist(),
            'image': [None] * len(url_sample),
            'labels': url_sample[self.url_label_col].tolist()
        }
        
        # Image only
        image_sample = self.stratified_sample_images(self.target_size)
        datasets['image_only'] = {
            'text': [None] * len(image_sample),
            'url': [None] * len(image_sample),
            'image': [img['path'] for img in image_sample],
            'labels': [img['label'] for img in image_sample]
        }
        
        # 2. Pairwise combinations
        print("Creating pairwise combination datasets...")
        
        # Text + URL
        text_url_sample = self.create_aligned_sample(['text', 'url'])
        datasets['text_url'] = text_url_sample
        
        # Text + Image  
        text_image_sample = self.create_aligned_sample(['text', 'image'])
        datasets['text_image'] = text_image_sample
        
        # URL + Image
        url_image_sample = self.create_aligned_sample(['url', 'image'])
        datasets['url_image'] = url_image_sample
        
        # 3. All modalities
        print("Creating full multimodal dataset...")
        all_modalities_sample = self.create_aligned_sample(['text', 'url', 'image'])
        datasets['all_modalities'] = all_modalities_sample
        
        # Print summary
        print(f"\n{'='*50}")
        print("DATASET GENERATION SUMMARY")
        print(f"{'='*50}")
        for name, data in datasets.items():
            label_dist = Counter(data['labels'])
            print(f"{name:15}: {len(data['labels'])} samples, {dict(label_dist)}")
        
        return datasets
    
    def create_aligned_sample(self, modalities):
        """Create aligned samples across multiple modalities with same labels"""
        # First, determine the label distribution we want to maintain
        # Use the most balanced distribution among available datasets
        
        text_labels = self.text_df[self.text_label_col].value_counts()
        url_labels = self.url_df[self.url_label_col].value_counts()
        image_label_list = [img['label'] for img in self.image_files]
        image_labels = pd.Series(image_label_list).value_counts()
        
        # Use proportions from the most balanced dataset
        all_distributions = [text_labels/len(self.text_df), url_labels/len(self.url_df), 
                           image_labels/len(self.image_files)]
        
        # Choose distribution with smallest difference between classes (most balanced)
        balance_scores = [abs(dist.iloc[0] - dist.iloc[1]) for dist in all_distributions]
        best_dist = all_distributions[np.argmin(balance_scores)]
        
        # Sample data for each modality
        result = {
            'text': [],
            'url': [], 
            'image': [],
            'labels': []
        }
        
        for label in best_dist.index:
            n_samples_label = int(self.target_size * best_dist[label])
            
            for i in range(n_samples_label):
                current_label = label
                
                # Sample from each required modality
                if 'text' in modalities:
                    text_sample = self.text_df[self.text_df[self.text_label_col] == current_label].sample(1)
                    result['text'].append(text_sample[self.text_col].iloc[0])
                else:
                    result['text'].append(None)
                
                if 'url' in modalities:
                    url_sample = self.url_df[self.url_df[self.url_label_col] == current_label].sample(1)
                    result['url'].append(url_sample[self.url_col].iloc[0])
                else:
                    result['url'].append(None)
                
                if 'image' in modalities:
                    available_images = [img for img in self.image_files if img['label'] == current_label]
                    if available_images:
                        img_sample = random.choice(available_images)
                        result['image'].append(img_sample['path'])
                    else:
                        result['image'].append(None)
                else:
                    result['image'].append(None)
                
                result['labels'].append(current_label)
        
        # Shuffle the results
        indices = list(range(len(result['labels'])))
        random.shuffle(indices)
        
        for key in result:
            result[key] = [result[key][i] for i in indices]
        
        return result
    
    def save_datasets(self, datasets, output_dir='mixed_datasets'):
        """Save generated datasets to files"""
        os.makedirs(output_dir, exist_ok=True)
        
        for name, data in datasets.items():
            # Create DataFrame
            df = pd.DataFrame({
                'text': data['text'],
                'url': data['url'], 
                'image_path': data['image'],
                'label': data['labels']
            })
            
            # Save to CSV
            output_path = os.path.join(output_dir, f'{name}.csv')
            df.to_csv(output_path, index=False)
            print(f"Saved {name} dataset to {output_path}")
    
    def create_train_test_splits(self, datasets, test_size=0.2):
        """Create train/test splits for all datasets"""
        split_datasets = {}
        
        for name, data in datasets.items():
            # Convert to arrays for sklearn
            X = list(zip(data['text'], data['url'], data['image']))
            y = data['labels']
            
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=test_size, random_state=42, stratify=y
            )
            
            # Convert back to dictionary format
            train_data = {
                'text': [x[0] for x in X_train],
                'url': [x[1] for x in X_train],
                'image': [x[2] for x in X_train],
                'labels': y_train
            }
            
            test_data = {
                'text': [x[0] for x in X_test],
                'url': [x[1] for x in X_test], 
                'image': [x[2] for x in X_test],
                'labels': y_test
            }
            
            split_datasets[name] = {
                'train': train_data,
                'test': test_data
            }
        
        return split_datasets

# Example usage
def main():
    """Example usage of the dataset mixer"""
    
    # Initialize mixer
    mixer = MultimodalDatasetMixer(
        text_csv_path='text_data/merged_enron.csv',
        url_csv_path='url_data/malicious_phish_cleaned.csv', 
        image_folder_path='image_data',  # Should contain spam/ and ham/ folders
        target_size=1739
    )
    
    # Load datasets
    mixer.load_datasets(
        text_col='text',      # Adjust column names as needed
        text_label_col='Label',
        url_col='url',
        url_label_col='label'
    )
    
    # Generate mixed datasets
    mixed_datasets = mixer.generate_mixed_datasets()
    
    # Create train/test splits
    split_datasets = mixer.create_train_test_splits(mixed_datasets)
    
    # Save datasets
    mixer.save_datasets(mixed_datasets)
    
    print("\nDataset mixing completed!")
    print("Available datasets:", list(mixed_datasets.keys()))
    
    return mixed_datasets, split_datasets

if __name__ == "__main__":
    datasets, splits = main()

In [3]:
import joblib

text_lemmatized_tfidf = joblib.load('./text_data/enron_lemmatized_tfidf.pkl')
text_X = text_lemmatized_tfidf['features']
text_y = text_lemmatized_tfidf['labels']

In [4]:
image_X = np.load('./image_data/data_X.npy')
image_y = np.load('./image_data/data_y.npy')

In [5]:
import pandas as pd
df = pd.read_csv("./url_data/malicious_phish_preprocessed.csv")
url_X = df.drop(columns=["url", "type", "label"])
url_y = df["label"]
del df

In [6]:
url_X = pd.get_dummies(url_X, columns=["tld"], drop_first=True)

In [7]:
mmc = MultimodalClassifier()

In [9]:
mmc.train(text_data=text_X, text_labels=text_y, image_data=image_X, image_labels=image_y, url_data=url_X, url_labels=url_y)

Begin training text
Text model accuracy: 0.9862
Begin training image
Epoch 1/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 47s 566ms/step - accuracy: 0.8303 - loss: 0.3243 - val_accuracy: 0.7126 - val_loss: 2.2563
Epoch 2/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 25s 535ms/step - accuracy: 0.9875 - loss: 0.0435 - val_accuracy: 0.8276 - val_loss: 2.0030
Epoch 3/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 25s 523ms/step - accuracy: 0.9946 - loss: 0.0111 - val_accuracy: 0.8659 - val_loss: 1.6638
Epoch 4/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 26s 560ms/step - accuracy: 0.9971 - loss: 0.0104 - val_accuracy: 0.8084 - val_loss: 3.1150
Epoch 5/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 25s 540ms/step - accuracy: 1.0000 - loss: 0.0019 - val_accuracy: 0.8314 - val_loss: 2.9526
Epoch 6/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 26s 546ms/step - accuracy: 1.0000 - loss: 7.0213e-04 - val_accuracy: 0.8621 - val_loss: 2.2645
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 226ms/step
Image model accuracy: 0.8659
Begin training URL
URL model accuracy: 0.9061


In [10]:
text_X_test, _, text_y_test, _ = train_test_split(text_X, text_y, random_state=42, train_size=2000)

In [11]:
url_X_test, _, url_y_test, _ = train_test_split(url_X, url_y, random_state=42, train_size=2000)

In [12]:
preds = mmc.predict(text_data=text_X_test, url_data=url_X_test)

In [14]:
from sklearn.metrics import classification_report

print(preds)
print(len(text_y_test))
print(len(url_y_test))

print(classification_report(url_y_test, preds))

[[1 0]
 [1 0]
 [1 0]
 ...
 [1 0]
 [1 0]
 [0 0]]
2000
2000


ValueError: Classification metrics can't handle a mix of binary and multilabel-indicator targets